# Descargador web de sentencias del STJ Jalisco (respaldo)

Copia sin modificar de las celdas de descarga del notebook original del proyecto (commit `8d88a36`).
Instrucciones de uso en `descarga_datos/README.md`.


In [ ]:
# FASE 0: Webscrapping
import os
import requests
import unicodedata
import re
import time
import json

def slugify(value, allow_unicode=False):
    """
    Convierte un string en un "slug" seguro para nombres de archivo.
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize('NFKC', value)
    else:
        value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub(r'[^\w\s-]', '', value.lower())
    return re.sub(r'[-\s]+', '-', value).strip('-_')

def setup_directory(folder_path):
    """
    Asegura que el directorio de destino exista y se mueva a él.
    """
    if not os.path.exists(folder_path):
        try:
            os.makedirs(folder_path)
            print(f"Carpeta creada en: {folder_path}")
        except OSError as e:
            print(f"Error al crear la carpeta {folder_path}: {e}")
            return False
    
    try:
        os.chdir(folder_path)
        print(f"Directorio de trabajo actual: {os.getcwd()}")
        return True
    except OSError as e:
        print(f"Error al cambiar al directorio {folder_path}: {e}")
        return False

def get_total_pages(api_url):
    """
    Obtiene el número total de páginas de la API.
    """
    try:
        print("Obteniendo número total de páginas...")
        page_response = requests.get(api_url)
        page_response.raise_for_status() # Verificar si hay errores
        last_page = page_response.json()['data']['tocas']['last_page']
        print(f"Total de páginas a procesar: {last_page}")
        return last_page
    except requests.RequestException as e:
        print(f"Error al obtener la paginación: {e}")
        return None
    except KeyError:
        print("Error: No se pudo encontrar 'last_page' en la respuesta de la API.")
        return None
    except json.JSONDecodeError:
        print("Error: La respuesta de la API no es un JSON válido.")
        return None

def download_sentencias(last_page, api_base_url):
    """
    Descarga todas las sentencias página por página.
    """
    for page_number in range(1, last_page + 1):
        print(f"\n--- Procesando página {page_number} de {last_page} ---")
        
        current_page_url = f"{api_base_url}/tocas?page={page_number}"
        
        try:
            page = requests.get(current_page_url)
            
            # Manejo de rate limiting (Error 429)
            if page.status_code == 429:
                retry_after = int(page.headers.get('retry-after', 30))
                print(f"Rate limit alcanzado. Esperando {retry_after} segundos...")
                time.sleep(retry_after)
                page = requests.get(current_page_url) # Reintentar
            
            page.raise_for_status()
            tocas_data = page.json()['data']['tocas']['data']
            
        except requests.RequestException as e:
            print(f"Error al obtener la página {page_number}: {e}. Saltando esta página.")
            continue
        except (KeyError, json.JSONDecodeError):
            print(f"Error de formato en la respuesta de la página {page_number}. Saltando.")
            continue

        # Iterar sobre cada sentencia (toca) en la página
        for toca in tocas_data:
            try:
                # --- 1. Validar y construir el nombre del archivo ---
                toca_id = toca.get('id')
                materia_data = toca.get('materia_data')
                delito_data = toca.get('delito_data')
                fecha = toca.get('fecha_publicacion') # Ej: "2024-01-08-103634"

                # Validar metadatos esenciales ANTES de acceder a sub-claves
                if not all([toca_id, materia_data, delito_data, fecha]):
                    print(f"Error: Faltan metadatos esenciales (ID, materia, delito o fecha) para Toca ID {toca_id}. Se omite.")
                    continue
                
                # Acceso seguro a sub-claves
                materia = materia_data.get('nombre')
                delito = delito_data.get('nombre')

                # Validar que las sub-claves también existan
                if not all([materia, delito]):
                    print(f"Error: Faltan 'nombre' en materia/delito para Toca ID {toca_id}. Se omite.")
                    continue
                
                # Construir la lista de partes del nombre
                partes_nombre = [fecha, materia, delito]
                
                # Añadir partes opcionales de forma segura
                accion_especifica = toca.get("accion_especifica_data")
                if accion_especifica and accion_especifica.get('nombre'):
                    partes_nombre.append(accion_especifica['nombre'])
                
                accion_simultanea = toca.get("accion_simultanea_data")
                if accion_simultanea and accion_simultanea.get('nombre'):
                    partes_nombre.append(accion_simultanea['nombre'])
                
                # Crear el nombre de archivo base
                toca_file_filename = "_".join(partes_nombre)
                
                # Limpiar y crear el nombre de archivo final
                filename = slugify(toca_file_filename) + '.pdf'
                
                # --- 2. Descargar y Guardar (Solo si el nombre fue exitoso) ---
                if not os.path.exists(filename):
                    # toca_id ya se obtuvo y validó arriba
                    toca_file_response = requests.get(f"{api_base_url}/toca/{toca_id}/file")
                    toca_file_response.raise_for_status() # Asegurarse que la descarga del PDF fue exitosa
                    
                    # Guardar el archivo
                    with open(filename, 'wb') as f:
                        f.write(toca_file_response.content)
                    print(f"Descargado: {filename}")
                    
                else:
                    print(f"Archivo ya existe: {filename}, se omite descarga.")
            
            except (TypeError, KeyError, AttributeError) as e_data:
                # Captura errores si 'materia_data' es None, o 'nombre' no existe, etc.
                print(f"Error de metadatos procesando Toca ID {toca.get('id', 'N/A')}: {e_data}. Se omite.")
            except requests.RequestException as e_req:
                # Captura errores de red (404, 500) al descargar el PDF
                print(f"Error de red al descargar Toca ID {toca.get('id', 'N/A')}: {e_req}. Se omite.")
            except Exception as e_other:
                # Captura cualquier otro error inesperado
                print(f"Error inesperado procesando Toca ID {toca.get('id', 'N/A')}: {e_other}. Se omite.")

    print("\n--- Proceso de descarga completado ---")

def main():
    # --- 1. Configuración ---
    # Ruta de destino para las sentencias (basada en tu celda de "manual_path")
    TARGET_FOLDER = '/home/nacho/Documents/diplomado_ia_cic/modulo_llm_api_tarea/prueba_dataset'
    API_BASE_URL = "https://publica-sentencias-backend.stjjalisco.gob.mx"
    PAGINATION_URL = f"{API_BASE_URL}/tocas?page=1&sala_id=&numero=&periodo=&derivado=&delito_id=&materia_id=&nombre_delito=&accion_especifica_id=&accion_simultanea_id="

    # --- 2. Preparar Directorio ---
    if not setup_directory(TARGET_FOLDER):
        return # Salir si no se puede configurar el directorio

    # --- 3. Obtener Paginación ---
    last_page = get_total_pages(PAGINATION_URL)
    if last_page is None:
        return # Salir si no se puede obtener la paginación

    # --- 4. Iniciar Descarga ---
    download_sentencias(last_page, API_BASE_URL)

if __name__ == "__main__":
    main()



In [ ]:
# FASE 0: Webscrapping (Dominio frontend corregido + Detección automática en Jupyter)
import os
import unicodedata
import re
import asyncio
from playwright.async_api import async_playwright

def slugify(value, allow_unicode=False):
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize('NFKC', value)
    else:
        value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub(r'[^\w\s-]', '', value.lower())
    return re.sub(r'[-\s]+', '-', value).strip('-_')

def setup_directory(folder_path):
    if not os.path.exists(folder_path):
        try:
            os.makedirs(folder_path)
            print(f"Carpeta creada en: {folder_path}", flush=True)
        except OSError as e:
            print(f"Error al crear la carpeta {folder_path}: {e}", flush=True)
            return False
    try:
        os.chdir(folder_path)
        print(f"Directorio de trabajo actual: {os.getcwd()}", flush=True)
        return True
    except OSError as e:
        print(f"Error al cambiar al directorio {folder_path}: {e}", flush=True)
        return False

async def main():
    TARGET_FOLDER = '/home/nacho/Documents/diplomado_ia_cic/modulo_llm_api_tarea/prueba_dataset'
    ORIGIN_URL = "https://publicacionsentencias.stjjalisco.gob.mx"
    FRONTEND_URL = f"{ORIGIN_URL}/sentencias"
    API_BASE_URL = "https://publica-sentencias-backend.stjjalisco.gob.mx"

    if not setup_directory(TARGET_FOLDER):
        return

    captured_headers = {
        "Accept": "application/json, text/plain, */*",
        "Origin": ORIGIN_URL,
        "Referer": f"{ORIGIN_URL}/"
    }
    sesion_verificada = asyncio.Event()

    def intercept_request(request):
        if API_BASE_URL in request.url and "/tocas" in request.url:
            for k, v in request.headers.items():
                if k.lower() not in ["host", "content-length", "connection"]:
                    captured_headers[k] = v

    def intercept_response(response):
        # Detecta automáticamente cuando el navegador carga la lista de sentencias con éxito (HTTP 200)
        if API_BASE_URL in response.url and "/tocas" in response.url and response.status == 200:
            sesion_verificada.set()

    print("Iniciando navegador Chromium...", flush=True)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()
        page.on("request", intercept_request)
        page.on("response", intercept_response)

        print(f"Abriendo {FRONTEND_URL}...", flush=True)
        await page.goto(FRONTEND_URL, wait_until="domcontentloaded", timeout=60000)

        print(
            "-> En la ventana de Chromium: resuelve el reCAPTCHA (o haz clic en 'Buscar') "
            "hasta que aparezca la tabla de sentencias...",
            flush=True
        )
        await sesion_verificada.wait()
        print("¡Sesión verificada! Iniciando descargas...", flush=True)

        async def fetch_api_with_retry(url):
            while True:
                response = await context.request.get(url, headers=captured_headers, timeout=30000)

                if response.status == 403 and "require_recaptcha" in (await response.text()):
                    print("\n[!] El servidor solicitó verificar reCAPTCHA nuevamente.", flush=True)
                    print("-> Ve a la ventana de Chromium y resuelve el CAPTCHA para continuar automáticamente...", flush=True)
                    sesion_verificada.clear()
                    await page.reload(wait_until="domcontentloaded")
                    await sesion_verificada.wait()
                    print("¡Re-verificado! Retomando descarga...", flush=True)
                    continue

                if response.status == 429:
                    print("Rate limit (429). Esperando 30 segundos...", flush=True)
                    await asyncio.sleep(30)
                    continue

                return response

        # 1. Obtener total de páginas
        pagination_url = (
            f"{API_BASE_URL}/tocas?page=1&sala_id=&numero=&periodo="
            "&derivado=&delito_id=&materia_id=&nombre_delito="
            "&accion_especifica_id=&accion_simultanea_id="
        )

        print("Obteniendo número total de páginas...", flush=True)
        first_res = await fetch_api_with_retry(pagination_url)
        if first_res.status != 200:
            print(f"Error inesperado ({first_res.status}): {(await first_res.text())[:300]}", flush=True)
            await browser.close()
            return

        first_json = await first_res.json()
        last_page = first_json['data']['tocas']['last_page']
        print(f"Total de páginas a procesar: {last_page}", flush=True)

        # 2. Descargar página por página
        for page_number in range(1, last_page + 1):
            print(f"\n--- Procesando página {page_number} de {last_page} ---", flush=True)
            current_page_url = (
                f"{API_BASE_URL}/tocas?page={page_number}&sala_id=&numero=&periodo="
                "&derivado=&delito_id=&materia_id=&nombre_delito="
                "&accion_especifica_id=&accion_simultanea_id="
            )

            res_page = await fetch_api_with_retry(current_page_url)
            if res_page.status != 200:
                print(f"Error en página {page_number} (Status {res_page.status}). Saltando.", flush=True)
                continue

            page_json = await res_page.json()
            tocas_data = page_json['data']['tocas']['data']

            for toca in tocas_data:
                try:
                    toca_id = toca.get('id')
                    materia_data = toca.get('materia_data')
                    delito_data = toca.get('delito_data')
                    fecha = toca.get('fecha_publicacion')

                    if not all([toca_id, materia_data, delito_data, fecha]):
                        continue

                    materia = materia_data.get('nombre')
                    delito = delito_data.get('nombre')
                    if not all([materia, delito]):
                        continue

                    partes_nombre = [str(toca_id), fecha, materia, delito]

                    accion_especifica = toca.get("accion_especifica_data")
                    if accion_especifica and accion_especifica.get('nombre'):
                        partes_nombre.append(accion_especifica['nombre'])

                    accion_simultanea = toca.get("accion_simultanea_data")
                    if accion_simultanea and accion_simultanea.get('nombre'):
                        partes_nombre.append(accion_simultanea['nombre'])

                    filename = slugify("_".join(partes_nombre)) + '.pdf'

                    if not os.path.exists(filename):
                        pdf_url = f"{API_BASE_URL}/toca/{toca_id}/file"
                        pdf_res = await fetch_api_with_retry(pdf_url)

                        if pdf_res.status == 200:
                            pdf_bytes = await pdf_res.body()
                            with open(filename, 'wb') as f:
                                f.write(pdf_bytes)
                            print(f"Descargado: {filename}", flush=True)
                            await asyncio.sleep(1.5)
                        else:
                            print(f"Error {pdf_res.status} al descargar PDF de Toca ID {toca_id}", flush=True)
                    else:
                        print(f"Archivo ya existe: {filename}, se omite descarga.", flush=True)

                except Exception as e:
                    print(f"Error procesando Toca ID {toca.get('id', 'N/A')}: {e}", flush=True)

            await asyncio.sleep(2)

        await browser.close()
        print("\n--- Proceso de descarga completado ---", flush=True)

await main()